In [1]:
import numpy as np
import sklearn as sk
import pandas as pd

In [2]:
df = pd.read_csv('cattle_data_train.csv')
print(len(df))


210000


Data Cleaning

In [3]:
labels = df["Milk_Yield_L"]

data = df.drop(columns=["Cattle_ID","Date","Farm_ID","Milk_Yield_L"])

# data = data.dropna() # Just drop all NaNs for now
# print(data.head())

categorical_cols = [c for c in data.columns if data[c].dtype == 'object']
numeric_cols = [c for c in data.columns if c not in categorical_cols]

# print(categorical_cols)
# print(numeric_cols)

selected_cols = categorical_cols[:]+numeric_cols[:]
selected_cols.append("Milk_Yield_L")
potential_data = df[selected_cols]
temp = sk.compose.ColumnTransformer(transformers=[("categorical",sk.preprocessing.OneHotEncoder(handle_unknown='ignore'),categorical_cols)],remainder="passthrough")
cpy = temp.fit_transform(potential_data)
print(pd.DataFrame(cpy,columns=temp.get_feature_names_out()).corr()["remainder__Milk_Yield_L"])




categorical__Breed_ Brown Swiss                 -0.001673
categorical__Breed_Brown Swiss                  -0.001069
categorical__Breed_Brown Swiss                  -0.002514
categorical__Breed_Guernsey                     -0.000333
categorical__Breed_Holstein                      0.001445
categorical__Breed_Holstien                      0.001963
categorical__Breed_Jersey                       -0.000510
categorical__Climate_Zone_Arid                   0.002194
categorical__Climate_Zone_Continental           -0.000920
categorical__Climate_Zone_Mediterranean         -0.001282
categorical__Climate_Zone_Subtropical           -0.000215
categorical__Climate_Zone_Temperate             -0.000323
categorical__Climate_Zone_Tropical               0.000545
categorical__Management_System_Extensive         0.002045
categorical__Management_System_Intensive         0.000791
categorical__Management_System_Mixed            -0.001226
categorical__Management_System_Pastoral         -0.000370
categorical__M

Remove features which have little correlation with milk yield

In [4]:
# Dropping all columns with less than .01 correlation with Milk_Yield_L
remove_cols = ["Breed","Climate_Zone","Management_System","Feeding_Frequency","Resting_Hours","Humidity_percent"
               ,"HS_Vaccine","BQ_Vaccine","BVD_Vaccine","Body_Condition_Score"]

categorical_cols=[x for x in categorical_cols if x not in remove_cols]
numeric_cols=[x for x in numeric_cols if x not in remove_cols]
print(categorical_cols)
print(numeric_cols)

['Lactation_Stage']
['Age_Months', 'Weight_kg', 'Parity', 'Days_in_Milk', 'Feed_Quantity_kg', 'Water_Intake_L', 'Rumination_Time_hrs', 'Ambient_Temperature_C', 'Anthrax_Vaccine', 'IBR_Vaccine', 'Rabies_Vaccine', 'Previous_Week_Avg_Yield', 'Milking_Interval_hrs', 'Feed_Quantity_lb', 'Mastitis']


In [4]:
numeric_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='median')),
    ('scaler', sk.preprocessing.StandardScaler())
])


categorical_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='most_frequent')),
    ('onehot', sk.preprocessing.OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = sk.compose.ColumnTransformer(transformers=[("numeric",numeric_transformer,numeric_cols),
("categorical",categorical_transformer,categorical_cols)])

pca = sk.decomposition.PCA()
nys = sk.kernel_approximation.Nystroem(n_components=1000)
ridge = sk.linear_model.LassoCV()
pipeline = sk.pipeline.Pipeline(steps=[("pre",preprocessor),("pca",pca),("nys",nys),("ridge",ridge)])
#"pca__n_components":[x/100 for x in range(80,100,5)],
params = {"pca__n_components":[.9],"nys__kernel":["linear","rbf","poly","sigmoid"]}

grid = sk.model_selection.GridSearchCV(pipeline,param_grid=params,scoring="neg_root_mean_squared_error",n_jobs=-5,verbose=3)
grid.fit(data,labels)
print(grid.best_score_)
print(grid.best_params_)
# score = sk.model_selection.cross_val_score(pipeline,data,labels,scoring="neg_root_mean_squared_error")
# print(score)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
-4.295295319369077
{'nys__kernel': 'sigmoid', 'pca__n_components': 0.9}


Build model with best params

In [ ]:
model = grid
test = pd.read_csv('cattle_data_test.csv')
inputs = test.drop(columns=["Cattle_ID","Date","Farm_ID"])
predictions = model.predict(inputs)
test["Milk_Yield_L"]=predictions
test[["Cattle_ID","Milk_Yield_L"]].to_csv("./results.csv",index=False)

In [ ]:
import pickle
pickle.dump(model,open("./final_model.sav","wb"))